In [ ]:
# [셀 1] 모델 로드 및 초기화
!pip install -q controlnet-aux diffusers transformers accelerate

import torch
from controlnet_aux import OpenposeDetector
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. OpenPose 관절 추출기 로드
processor = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

# 2. ControlNet 및 이미지 생성 파이프라인 로드
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

if device == "cuda":
    pipe.enable_model_cpu_offload()

print("✅ 모델 로드 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 10.6 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segmen

annotator/ckpts/body_pose_model.pth: reconstructing file:   0%|          |  0.00B /  209MB            

annotator/ckpts/body_pose_model.pth: downloading bytes:           |  0.00B            

annotator/ckpts/hand_pose_model.pth: reconstructing file:   0%|          |  0.00B /  147MB            

...

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ 모델 로드 완료


In [ ]:
# [셀 2] 참조 이미지 로드 및 포즈 추출
from diffusers.utils import load_image

# 참조 포즈 이미지 경로 또는 URL 설정
input_image_path = "./samples/pose_01.png"  # Colab에 업로드한 이미지 경로 또는 URL

# 이미지 불러오기
input_image = load_image(input_image_path)

# OpenPose로 관절 포즈(Pose) 추출
pose_image = processor(input_image)

# 추출된 포즈 이미지 확인/저장
pose_image.save("./samples/extracted_pose.png")
display(pose_image)  # Colab 출력창에 포즈 이미지 표시

In [ ]:
# [셀 2] 참조 이미지 로드 및 포즈 추출
from diffusers.utils import load_image

# 참조 포즈 이미지 경로 또는 URL 설정
input_image_path = "./samples/pose_02.png"  # Colab에 업로드한 이미지 경로 또는 URL

# 이미지 불러오기
input_image = load_image(input_image_path)

# OpenPose로 관절 포즈(Pose) 추출
pose_image = processor(input_image)

# 추출된 포즈 이미지 확인/저장
pose_image.save("./samples/extracted_pose.png")
display(pose_image)  # Colab 출력창에 포즈 이미지 표시

In [ ]:
# [셀 3] 프롬프트 조건 입력 및 이미지 생성
prompt = "graceful female ice skater, sparkling skating costume, elegant movement, magical ice rink, snow particles, frozen crystals, flowing hair, winter fantasy atmosphere, cinematic lighting, photorealistic"
negative_prompt = "lowres, blurry, bad anatomy, deformed, extra limbs, extra fingers, bad hands, bad feet, broken legs, cropped, watermark"
# 시드값 고정
generator = torch.Generator(device=device).manual_seed(42)

# ControlNet으로 이미지 생성
generated_image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=pose_image,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

display(generated_image)  # Colab 출력창에 생성 결과 표시
